# Testing the Prediction Module Machinery

This notebook tests the functionality of the prediction module for protein engineering models, particularly focusing on the data loading functionality.

In [4]:
%load_ext autoreload
%autoreload 2
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Configure logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

from prediction import get_available_datasets, get_proteingym_dataset

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Importing the Prediction Module

First, let's try to import our prediction module functions.

# Import the prediction module
try:
    from prediction import get_available_proteingym_datasets, get_proteingym_dataset
    print("Successfully imported the prediction module functions!")
except Exception as e:
    print(f"Error importing the prediction module: {e}")

In [5]:
# Define paths
from prediction.data import MODULE_DIR, DATA_DIR, DMS_DIR, EMBEDDINGS_DIR, NATURALNESS_DIR, DMS_METADATA_FILE

print(f"Module directory: {MODULE_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"DMS directory: {DMS_DIR}")
print(f"Embeddings directory: {EMBEDDINGS_DIR}")
print(f"Naturalness directory: {NATURALNESS_DIR}")
print(f"DMS metadata file: {DMS_METADATA_FILE}")

Module directory: /Users/jacobroberts/git/foldy/backend/src/prediction
Data directory: /Users/jacobroberts/git/foldy/backend/src/prediction/data
DMS directory: /Users/jacobroberts/git/foldy/backend/src/prediction/data/DMS_ProteinGym_substitutions
Embeddings directory: /Users/jacobroberts/git/foldy/backend/src/prediction/data/embeddings
Naturalness directory: /Users/jacobroberts/git/foldy/backend/src/prediction/data/naturalness
DMS metadata file: /Users/jacobroberts/git/foldy/backend/src/prediction/data/DMS_substitutions.csv


In [6]:
# Check if directories and metadata file exist
print(f"Data directory exists: {os.path.exists(DATA_DIR)}")
print(f"DMS directory exists: {os.path.exists(DMS_DIR)}")
print(f"Embeddings directory exists: {os.path.exists(EMBEDDINGS_DIR)}")
print(f"Naturalness directory exists: {os.path.exists(NATURALNESS_DIR)}")
print(f"DMS metadata file exists: {os.path.exists(DMS_METADATA_FILE)}")

Data directory exists: True
DMS directory exists: True
Embeddings directory exists: True
Naturalness directory exists: True
DMS metadata file exists: True


In [7]:
# List embedding files
embedding_files = list(Path(EMBEDDINGS_DIR).glob("*.csv"))
print(f"Found {len(embedding_files)} embedding files")
for i, file in enumerate(embedding_files[:5]):
    print(f"  {i+1}. {file.name}")
if len(embedding_files) > 5:
    print(f"  ... and {len(embedding_files) - 5} more")

Found 2 embedding files
  1. BLAT_ECOLX_Stiffler_2015_embedding_300m.csv
  2. A0A140D2T1_ZIKV_Sourisseau_2019_embedding_300m.csv


In [8]:
# List naturalness files
naturalness_files = list(Path(NATURALNESS_DIR).glob("*.csv"))
print(f"Found {len(naturalness_files)} naturalness files")
for i, file in enumerate(naturalness_files[:5]):
    print(f"  {i+1}. {file.name}")
if len(naturalness_files) > 5:
    print(f"  ... and {len(naturalness_files) - 5} more")

Found 1 naturalness files
  1. A0A140D2T1_ZIKV_Sourisseau_2019_naturalness_650m.csv


## Testing get_available_datasets

Now let's test the `get_available_datasets` function with specific embedding and naturalness models.

In [9]:

embedding_model_id = '300m'
naturalness_model_id = '600m'

print(f"Testing with embedding model: {embedding_model_id} and naturalness model: {naturalness_model_id}")
available_datasets = get_available_proteingym_datasets(embedding_model_id, naturalness_model_id)

if not available_datasets.empty:
    print(f"Found {len(available_datasets)} available datasets")
    print("\nSample of available datasets:")
    display(available_datasets.head())
else:
    print("No datasets found with these models")

2025-03-15 14:44:54,180 - prediction.data - INFO - Loaded metadata for 217 DMS datasets
2025-03-15 14:44:54,198 - prediction.data - INFO - Found 3 datasets with embedding model '300m' and naturalness model '600m'


Testing with embedding model: 300m and naturalness model: 600m
Found 3 available datasets

Sample of available datasets:


,DMS_id,DMS_filename,UniProt_ID,taxon,source_organism,target_seq,seq_len,includes_multiple_mutants,DMS_total_number_mutants,DMS_number_single_mutants,...,raw_DMS_filename,raw_DMS_phenotype_name,raw_DMS_directionality,raw_DMS_mutant_column,weight_file_name,pdb_file,pdb_range,ProteinGym_version,raw_mut_offset,coarse_selection_type
0,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1_ZIKV_Sourisseau_2019.csv,A0A140D2T1_ZIKV,Virus,Zika virus (ZIKV),MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,3423,False,9576,9576,...,A0A140D2T1_ZIKV_Sourisseau_growth_2019.csv,effect,1,mutant,A0A140D2T1_ZIKV_theta_0.01.npy,A0A140D2T1_ZIKV.pdb,291-794,0.1,NaN,OrganismalFitness
23,BLAT_ECOLX_Stiffler_2015,BLAT_ECOLX_Stiffler_2015.csv,BLAT_ECOLX,Prokaryote,Escherichia coli,MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIE...,286,False,4996,4996,...,BLAT_ECOLX_Stiffler_2015.csv,2500,1,mutant,BLAT_ECOLX_theta_0.2.npy,BLAT_ECOLX.pdb,1-286,0.1,NaN,OrganismalFitness
124,PHOT_CHLRE_Chen_2023,PHOT_CHLRE_Chen_2023.csv,PHOT_CHLRE,Eukaryote,Chlamydomonas reinhardtii,AGLRHTFVVADATLPDCPLVYASEGFYAMTGYGPDEVLGHNARFLQ...,118,True,167529,2122,...,sb2c00662_si_001.xlsx,mean,1,mutant,PHOT_CHLRE_theta0.2_2023-08-07_b02.npy,PHOT_CHLRE.pdb,1-118,1.0,NaN,Activity


In [10]:
dms_id = available_datasets['DMS_id'].iloc[0]
    
print(f"Testing with dataset: {dms_id}, embedding model: {embedding_model_id}, naturalness model: {naturalness_model_id}")

try:
    naturalness_df, embedding_df, activity_df = get_proteingym_dataset(dms_id, embedding_model_id, naturalness_model_id)
    print(f"Successfully loaded datasets:")
    print(f"  - Naturalness data: {len(naturalness_df)} rows")
    print(f"  - Embedding data: {len(embedding_df)} rows")
    print(f"  - Activity data: {len(activity_df)} rows")
    
    # Display dataset summaries
    print("\nNaturalness DataFrame columns:")
    print(naturalness_df.columns.tolist())
    
    print("\nEmbedding DataFrame columns:")
    print(embedding_df.columns.tolist())
    
    print("\nActivity DataFrame columns:")
    print(activity_df.columns.tolist())
    
    # Display sample of each dataframe
    print("\nSample of naturalness data:")
    display(naturalness_df.head())
    
    print("\nSample of activity data:")
    display(activity_df.head())
    
    # Check embedding format
    print("\nEmbedding format:")
    embedding = embedding_df['embedding'].iloc[0]
    print(f"Type: {type(embedding)}")
    print(f"Shape: {embedding.shape if hasattr(embedding, 'shape') else 'N/A'}")
    print(f"First 5 values: {embedding[:5] if hasattr(embedding, '__getitem__') else 'N/A'}")
    
    # Check seq_id format in all dataframes
    print("\nSeq_id examples:")
    print(f"Naturalness first seq_id: {naturalness_df['seq_id'].iloc[0] if 'seq_id' in naturalness_df.columns else 'N/A'}")
    print(f"Embedding first seq_id: {embedding_df['seq_id'].iloc[0] if 'seq_id' in embedding_df.columns else 'N/A'}")
    print(f"Activity first seq_id: {activity_df['seq_id'].iloc[0] if 'seq_id' in activity_df.columns else 'N/A'}")
    
except Exception as e:
    print(f"Error loading dataset: {e}")

Testing with dataset: A0A140D2T1_ZIKV_Sourisseau_2019, embedding model: 300m, naturalness model: 600m


2025-03-15 14:45:18,564 - prediction.data - INFO - Loaded activity data for A0A140D2T1_ZIKV_Sourisseau_2019 with 9576 rows
2025-03-15 14:45:18,573 - prediction.data - INFO - Converted 'mutant' column to 'seq_id'
2025-03-15 14:45:20,463 - prediction.data - INFO - Loaded embeddings for A0A140D2T1_ZIKV_Sourisseau_2019 with 9577 rows
2025-03-15 14:45:20,472 - prediction.data - INFO - Loaded naturalness scores for A0A140D2T1_ZIKV_Sourisseau_2019 with 16632 rows


Error loading dataset: Naturalness file missing 'wt_marginal' column. Available columns: ['seq_id', 'probability']


# If we have datasets, visualize aspects of them
if 'naturalness_df' in locals() and 'embedding_df' in locals() and 'activity_df' in locals():
    # Plot 1: Embedding dimensionality
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    embedding = embedding_df['embedding'].iloc[0]
    plt.hist(embedding, bins=30, alpha=0.7)
    plt.title(f'Histogram of First Embedding Values\n(dimension={len(embedding) if hasattr(embedding, "__len__") else "unknown"})')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    
    # Plot 2: Naturalness distribution
    if 'naturalness' in naturalness_df.columns:
        plt.subplot(1, 2, 2)
        plt.hist(naturalness_df['naturalness'], bins=30, alpha=0.7)
        plt.title('Distribution of Naturalness Scores')
        plt.xlabel('Naturalness')
        plt.ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
    
    # If there is a clear target column for fitness or activity, visualize it
    possible_target_cols = ['fitness', 'score', 'activity', 'DMS_score', 'measurement']
    target_col = None
    
    for col in possible_target_cols:
        if col in activity_df.columns and activity_df[col].dtype in [np.float64, np.int64]:
            target_col = col
            break
    
    if target_col:
        plt.figure(figsize=(14, 5))
        
        # Plot target distribution
        plt.subplot(1, 2, 1)
        plt.hist(activity_df[target_col], bins=30, alpha=0.7)
        plt.title(f'Distribution of {target_col}')
        plt.xlabel(target_col)
        plt.ylabel('Frequency')
        
        # Checking if we can match naturalness data with activity data
        if 'seq_id' in activity_df.columns and 'seq_id' in naturalness_df.columns:
            # Merge datasets for plotting
            merged = activity_df[['seq_id', target_col]].merge(
                naturalness_df[['seq_id', 'naturalness']], on='seq_id', how='inner'
            )
            
            if not merged.empty:
                plt.subplot(1, 2, 2)
                plt.scatter(merged['naturalness'], merged[target_col], alpha=0.5)
                plt.title(f'Naturalness vs {target_col}')
                plt.xlabel('Naturalness')
                plt.ylabel(target_col)
                print(f"Merged {len(merged)} rows with both naturalness and {target_col} data")
            else:
                print("Could not merge naturalness with activity data - no matching seq_ids")
        
        plt.tight_layout()
        plt.show()
else:
    print("No datasets available for visualization")

In [ ]:
# If we have available datasets, test loading one
if 'available_datasets' in locals() and not available_datasets.empty:
    # Use the first available dataset for testing
    dms_id = available_datasets['DMS_id'].iloc[0]
    
    print(f"Testing with dataset: {dms_id}, embedding model: {embedding_model_id}, naturalness model: {naturalness_model_id}")
    
    try:
        dataset = get_proteingym_dataset(dms_id, embedding_model_id, naturalness_model_id)
        print(f"Successfully loaded dataset with {len(dataset)} rows")
        
        # Display dataset summary
        print("\nDataset columns:")
        print(dataset.columns.tolist())
        
        print("\nSample of dataset (first 5 rows, selected columns):")
        display_cols = [col for col in dataset.columns if col not in ['embedding']]
        display(dataset[display_cols].head())
        
        # Check embedding format
        print("\nEmbedding format:")
        embedding = dataset['embedding'].iloc[0]
        print(f"Type: {type(embedding)}")
        print(f"Shape: {embedding.shape}")
        print(f"First 5 values: {embedding[:5]}")
        
        # Check naturalness format
        if 'naturalness' in dataset.columns:
            print("\nNaturalness format:")
            print(f"Type: {type(dataset['naturalness'].iloc[0])}")
            print(f"Range: {dataset['naturalness'].min()} to {dataset['naturalness'].max()}")
            print(f"Mean: {dataset['naturalness'].mean():.4f}")
    except Exception as e:
      raise e
else:
    print("No available datasets to test loading")

Testing with dataset: A0A140D2T1_ZIKV_Sourisseau_2019, embedding model: 300m, naturalness model: 650m


2025-03-15 13:46:03,907 - prediction.data - INFO - Loaded DMS data for A0A140D2T1_ZIKV_Sourisseau_2019 with 9576 rows
2025-03-15 13:46:05,472 - prediction.data - INFO - Loaded embeddings for A0A140D2T1_ZIKV_Sourisseau_2019 with 9577 rows
2025-03-15 13:46:05,482 - prediction.data - INFO - Loaded naturalness scores for A0A140D2T1_ZIKV_Sourisseau_2019 with 13608 rows


KeyError: 'seq_id'

## Visualizing Data

If we successfully loaded a dataset, let's visualize some aspects of it.

In [ ]:
# If we have a dataset, visualize it
if 'dataset' in locals() and not dataset.empty:
    plt.figure(figsize=(14, 5))
    
    # Plot 1: Embedding dimensionality
    plt.subplot(1, 2, 1)
    embedding = dataset['embedding'].iloc[0]
    plt.hist(embedding, bins=30, alpha=0.7)
    plt.title(f'Histogram of First Embedding Values\n(dimension={len(embedding)})')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    
    # Plot 2: Naturalness distribution if available
    if 'naturalness' in dataset.columns:
        plt.subplot(1, 2, 2)
        plt.hist(dataset['naturalness'], bins=30, alpha=0.7)
        plt.title('Distribution of Naturalness Scores')
        plt.xlabel('Naturalness')
        plt.ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
    
    # If there is a clear target column for fitness or activity, visualize it
    possible_target_cols = ['fitness', 'score', 'activity', 'DMS_score', 'measurement']
    target_col = None
    
    for col in possible_target_cols:
        if col in dataset.columns and dataset[col].dtype in [np.float64, np.int64]:
            target_col = col
            break
    
    if target_col:
        plt.figure(figsize=(14, 5))
        
        # Plot target distribution
        plt.subplot(1, 2, 1)
        plt.hist(dataset[target_col], bins=30, alpha=0.7)
        plt.title(f'Distribution of {target_col}')
        plt.xlabel(target_col)
        plt.ylabel('Frequency')
        
        # Plot target vs naturalness if available
        if 'naturalness' in dataset.columns:
            plt.subplot(1, 2, 2)
            plt.scatter(dataset['naturalness'], dataset[target_col], alpha=0.5)
            plt.title(f'Naturalness vs {target_col}')
            plt.xlabel('Naturalness')
            plt.ylabel(target_col)
        
        plt.tight_layout()
        plt.show()
else:
    print("No dataset available for visualization")

## Conclusion

This notebook has tested the basic functionality of the prediction module's data loading capabilities. We've:

1. Confirmed that the module can be imported correctly
2. Explored the data directories and available files
3. Tested the `get_available_datasets` function
4. Tested the `get_proteingym_dataset` function
5. Visualized aspects of the loaded data

The infrastructure is now ready for implementing and testing machine learning models for protein engineering prediction tasks.